In [ ]:
import multiprocessing


In [3]:
import weakref

class B:
    def __init__(self, x):
        self.x = x
        
    def __repr__(self):
        return f'B({self.x})'

class A(B):
    q = []
    
    @classmethod
    def f(cls, ref):
        cls.q.remove(ref)
        
    @classmethod
    def g(cls, x):
        cls.q.append(weakref.ref(x, cls.f))
        
print(A.__qualname__)
print(B.__qualname__)

x = B(10)
print(x)

A.g(x)
print(A.q)
print(x)
print(A.q)
x = 5
print(A.q)

A
B
B(10)
[<weakref at 0x7f1a2c653f60; to 'B' at 0x7f1a3553d390>]
B(10)
[<weakref at 0x7f1a2c653f60; to 'B' at 0x7f1a3553d390>]
[]


In [25]:
def f(x, y):
    print(x, y)
    
    
f(1, 5, z=10)

TypeError: f() got an unexpected keyword argument 'z'

In [11]:
from time import time

time()

1705910445.2114437

In [24]:
class A:
    x = []
    
    def __init__(self):
        ...
        
    @classmethod
    def f(cls):
        cls.x.append(1)
        

class B(A):
    def __init__(self):
        ...
        
    @classmethod
    def f(cls):
        cls.x.append(1)


a = A()
print(a.x)
a.x.append(1)
print(a.x)
b = A()
print(a.x)
print(b.x)
c = B()
print(c.x)
B.f()
print(c.x)

[]
[1]
[1]
[1]
[1]
[1, 1]


In [8]:
from typing import Any


class A():
    def __getattr__(self, name):
        print(f"get attr {name}")
        return super().__getattr__(name)
    
    
    def __getattribute__(self, __name: str) -> Any:
        print(f"$ get attr {__name}")
        return getattr(self, __name)
    
    def __init__(self, name):
        self.name = name
        self._x = 5
        self.__y = 7
        self.z = 9
        print("init")
        print(self.name, self._x, self.__y, self.z)
        
        
a = A('R')
print(a.z)
print(a.__dict__)

init
$ get attr name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name
$ get attr _A__name


KeyboardInterrupt



In [5]:
class A:
    def f(self):
        print(self.__class__.__qualname__)


class B(A):
    pass


b = B()
b.f()

B


In [11]:
s = '{s}, {s}, {x}'
s.format(dict(x=1, s=7), s=9, x=8)

'9, 9, 8'

In [13]:
def f(args):
    def g(*args):
        print(args)
    g(*args)
    
    
    
f(None)

TypeError: __main__.f.<locals>.g() argument after * must be an iterable, not NoneType

In [15]:
x, y = ()
print(x, y)

ValueError: not enough values to unpack (expected 2, got 0)

In [4]:
f = lambda x: x
print(repr(f))

<function <lambda> at 0x7fefbc3a8f70>


In [241]:
import os
import random
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from functools import reduce


class StupidLinearModule(nn.Module):
    def __init__(self, inp_dim, oup_dim=None, double_w=False):
        super().__init__()
        self.double_w = double_w
        if self.double_w:
            if oup_dim is None:
                oup_dim = inp_dim
            self.w1 = nn.Linear(inp_dim, oup_dim)
            self.relu = nn.ReLU()
            self.w2 = nn.Linear(oup_dim, 1)
        else:
            self.w1 = nn.Linear(inp_dim, 1, bias=False)
    
    def forward(self, x):
        if self.double_w:
            return self.w2(self.relu(self.w1(x)))
        else:
            return self.w1(x)


inp_dim = 3
oup_dim = 10
double_w = False
MOD = 223
train_iters = 10000
debug = False
batch_size = 32
# model = StupidLinearModule(inp_dim).to('cuda')
model = StupidLinearModule(inp_dim, oup_dim, double_w).to('cuda')
optimizer = optim.AdamW(model.parameters(), lr=5e-3)
grad_acc_step = 1
mae_gamma = 0.9
mae_loss = 0
for i in tqdm(range(train_iters)):
    if debug:
        x = [random.randint(0, MOD-1) for _ in range(inp_dim-1)]
        label = reduce((lambda x, y: (x * y) % MOD), x)
        x += [label]
        x = torch.Tensor(x).to('cuda')
    else:
        x = torch.Tensor([[random.randint(0, MOD-1)/MOD for _ in range(inp_dim)] for __ in range(batch_size)]).to('cuda')
        label = torch.Tensor([reduce((lambda x, y: (int(MOD*x) + int(MOD*y))/MOD), _x) for _x in x]).to('cuda')
    loss = ((MOD*model(x).squeeze() - MOD*label)**2).mean()
    mae_loss = (1-mae_gamma)*mae_loss+mae_gamma*loss.item()
    print(f'iter {i},{"":6s}loss: {loss.item():.10f},{"":6s}mae_loss: {mae_loss:.10f}')
    loss.backward()
    if (i + 1) % grad_acc_step == 0:
        optimizer.step()
        optimizer.zero_grad()

  1%|          | 59/10000 [00:00<00:34, 292.15it/s]

iter 0,      loss: 136196.6875000000,      mae_loss: 122577.0187500000
iter 1,      loss: 116114.5468750000,      mae_loss: 116760.7940625000
iter 2,      loss: 135202.5937500000,      mae_loss: 133358.4137812500
iter 3,      loss: 120343.2890625000,      mae_loss: 121644.8015343750
iter 4,      loss: 114617.1015625000,      mae_loss: 115319.8715596875
iter 5,      loss: 153940.5937500000,      mae_loss: 150078.5215309688
iter 6,      loss: 141037.6562500000,      mae_loss: 141941.7427780969
iter 7,      loss: 125523.8906250000,      mae_loss: 127165.6758403097
iter 8,      loss: 117438.3593750000,      mae_loss: 118411.0910215310
iter 9,      loss: 153857.7187500000,      mae_loss: 150313.0559771531
iter 10,      loss: 118772.3671875000,      mae_loss: 121926.4360664653
iter 11,      loss: 140805.0000000000,      mae_loss: 138917.1436066465
iter 12,      loss: 110829.9218750000,      mae_loss: 113638.6440481647
iter 13,      loss: 117533.0390625000,      mae_loss: 117143.5995610665
it

  1%|          | 121/10000 [00:00<00:32, 301.95it/s]

iter 59,      loss: 69394.2109375000,      mae_loss: 70744.2070600381
iter 60,      loss: 83568.0625000000,      mae_loss: 82285.6769560038
iter 61,      loss: 66182.6562500000,      mae_loss: 67792.9583206004
iter 62,      loss: 60256.9414062500,      mae_loss: 61010.5430976850
iter 63,      loss: 65317.1875000000,      mae_loss: 64886.5230597685
iter 64,      loss: 67893.3671875000,      mae_loss: 67592.6827747269
iter 65,      loss: 66444.6328125000,      mae_loss: 66559.4378087227
iter 66,      loss: 77894.1406250000,      mae_loss: 76760.6703433723
iter 67,      loss: 81563.7500000000,      mae_loss: 81083.4420343372
iter 68,      loss: 60533.7148437500,      mae_loss: 62588.6875628087
iter 69,      loss: 71959.4062500000,      mae_loss: 71022.3343812809
iter 70,      loss: 53572.1796875000,      mae_loss: 55317.1951568781
iter 71,      loss: 61995.2382812500,      mae_loss: 61327.4339688128
iter 72,      loss: 64957.7460937500,      mae_loss: 64594.7148812563
iter 73,      loss: 

  2%|▏         | 183/10000 [00:00<00:32, 304.15it/s]

...

iter 9868,      loss: 0.2883716226,      mae_loss: 0.2799521035
iter 9869,      loss: 0.0368569791,      mae_loss: 0.0611664916
iter 9870,      loss: 0.1452721953,      mae_loss: 0.1368616250
iter 9871,      loss: 0.1184895486,      mae_loss: 0.1203267562
iter 9872,      loss: 0.1811547577,      mae_loss: 0.1750719576
iter 9873,      loss: 0.1997591704,      mae_loss: 0.1972904491
iter 9874,      loss: 0.1224115789,      mae_loss: 0.1298994659
iter 9875,      loss: 0.2105371803,      mae_loss: 0.2024734089
iter 9876,      loss: 0.1441725045,      mae_loss: 0.1500025950
iter 9877,      loss: 0.2186688185,      mae_loss: 0.2118021961
iter 9878,      loss: 0.1752711236,      mae_loss: 0.1789242309
iter 9879,      loss: 0.0838340223,      mae_loss: 0.0933430431
iter 9880,      loss: 0.1748988628,      mae_loss: 0.1667432809
iter 9881,      loss: 0.1235185564,      mae_loss: 0.1278410288
iter 9882,      loss: 0.2559377551,      mae_loss: 0.2431280825
iter 9883,      loss: 0.0893574804,     

100%|█████████▉| 9979/10000 [00:32<00:00, 304.66it/s]

iter 9930,      loss: 0.1641063690,      mae_loss: 0.1658758382
iter 9931,      loss: 0.1603156179,      mae_loss: 0.1608716399
iter 9932,      loss: 0.2401617914,      mae_loss: 0.2322327763
iter 9933,      loss: 0.0963068679,      mae_loss: 0.1098994587
iter 9934,      loss: 0.1182780340,      mae_loss: 0.1174401765
iter 9935,      loss: 0.1155596375,      mae_loss: 0.1157476914
iter 9936,      loss: 0.1559446305,      mae_loss: 0.1519249366
iter 9937,      loss: 0.2298752666,      mae_loss: 0.2220802336
iter 9938,      loss: 0.1491574794,      mae_loss: 0.1564497548
iter 9939,      loss: 0.1223909855,      mae_loss: 0.1257968624
iter 9940,      loss: 0.0902117789,      mae_loss: 0.0937702872
iter 9941,      loss: 0.1432642341,      mae_loss: 0.1383148394
iter 9942,      loss: 0.2847180963,      mae_loss: 0.2700777706
iter 9943,      loss: 0.1391447037,      mae_loss: 0.1522380104
iter 9944,      loss: 0.1121782362,      mae_loss: 0.1161842137
iter 9945,      loss: 0.2494567037,     

100%|██████████| 10000/10000 [00:32<00:00, 303.31it/s]

iter 9992,      loss: 0.0374495201,      mae_loss: 0.0554517196
iter 9993,      loss: 0.2347390056,      mae_loss: 0.2168102770
iter 9994,      loss: 0.1066111401,      mae_loss: 0.1176310538
iter 9995,      loss: 0.2814689875,      mae_loss: 0.2650851941
iter 9996,      loss: 0.1114577055,      mae_loss: 0.1268204544
iter 9997,      loss: 0.1088922620,      mae_loss: 0.1106850812
iter 9998,      loss: 0.1285849810,      mae_loss: 0.1267949910
iter 9999,      loss: 0.1022269502,      mae_loss: 0.1046837543


In [234]:
x = torch.Tensor([random.randint(0, MOD-1)/MOD for _ in range(inp_dim)]).to('cuda')
label = reduce((lambda x, y: (int(MOD*x) + int(MOD*y))/MOD), x)
pred = model(x)
print(x*MOD, label*MOD, f'{float(pred*MOD):10f}')

tensor([ 28., 167., 145.], device='cuda:0') 340.0 340.000244


In [235]:
for name, param in model.named_parameters():
    print(name, param)

w1.weight Parameter containing:
tensor([[0.9995, 1.0002, 0.9999]], device='cuda:0', requires_grad=True)
